In [1]:
# STEP 2.1: Install Unsloth
!pip install -q unsloth

In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Torch version:", torch.__version__)

GPU available: True
Torch version: 2.14.1+cu130


In [3]:
# ===== STEP 2.1 VALIDATION =====
import unsloth, torch
print("Unsloth version :", unsloth.__version__)
print("GPU available   :", torch.cuda.is_available())
print("Torch version   :", torch.__version__)
print("GPU name        :", torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"GPU memory      : {free/1e9:.1f} GB free of {total/1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1125 02:50:06.669000 2522 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1125 02:50:06.712000 2522 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth version : 2026.11.11
GPU available   : True
Torch version   : 2.14.1+cu130
GPU name        : Tesla T4
GPU memory      : 15.5 GB free of 15.6 GB


In [4]:
# STEP 2.2: Load Mistral 7B in 4-bit
from unsloth import FastLanguageModel
import torch

MAX_LEN = 1024   # from Step 1.6

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",  # pre-compressed Mistral
    max_seq_length=MAX_LEN,
    dtype=None,          # let Unsloth pick the right number format for the T4
    load_in_4bit=True,   # keep it compressed (4-bit)
)

# --- Checks ---
used = torch.cuda.memory_allocated() / 1e9
free, total = torch.cuda.mem_get_info()
print("\n===== STEP 2.2 CHECK =====")
print("Model loaded      :", type(model).__name__)
print("Loaded in 4-bit   :", getattr(model, "is_loaded_in_4bit", "unknown"))
print("Number format     :", model.dtype)
print(f"GPU memory used   : {used:.1f} GB")
print(f"GPU memory free   : {free/1e9:.1f} GB of {total/1e9:.1f} GB")

==((====))==  Unsloth 2026.11.11: Fast Mistral patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


===== STEP 2.2 CHECK =====
Model loaded      : MistralForCausalLM
Loaded in 4-bit   : True
Number format     : torch.float16
GPU memory used   : 4.1 GB
GPU memory free   : 11.3 GB of 15.6 GB


In [5]:
# ===== STEP 2.3: First question to Mistral =====
import time

FastLanguageModel.for_inference(model)   # switch Unsloth to "answering mode" (faster replies)

def ask(question, system_prompt=None, max_new_tokens=400):
    """Send one customer question to the model and return its answer."""
    messages = [{"role": "user", "content": question}]
    if system_prompt:   # optional extra instruction (used in Step 3)
        messages = [{"role": "system", "content": system_prompt}] + messages

    # 1) Put the question in Mistral's format: <s>[INST] question [/INST]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")

    # 2) Let the model write its answer
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,  # safety limit on answer length
        do_sample=False,                # always pick the most likely word, so same question = same answer
        pad_token_id=tokenizer.eos_token_id,
    )

    # 3) Keep only the NEW part (the answer), not the question we sent
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer, len(new_tokens)

# --- Try it ---
question = "I want to cancel my order, how do I do it?"
start = time.time()
answer, n_tokens = ask(question)
seconds = time.time() - start

print("CUSTOMER:", question)
print("\nMISTRAL (original, not trained yet):\n", answer)
print("\n===== STEP 2.3 CHECK =====")
print("Answer length (tokens):", n_tokens, "| limit was 400")
print("Stopped on its own    :", n_tokens < 400)
print(f"Time taken            : {seconds:.1f} seconds")

Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CUSTOMER: I want to cancel my order, how do I do it?

MISTRAL (original, not trained yet):
 To cancel an order, you typically need to follow these steps:

1. Locate your order: Check your email for the order confirmation, or log into your account on the website or app where you made the purchase.

2. Find the order details: Once you've located your order, find the order number and the item(s) you want to cancel.

3. Initiate the cancellation: Look for a "Cancel Order" or "Return/Exchange" option. This is usually found in the order details section.

4. Follow the prompts: The system will guide you through the cancellation process. You may be asked to provide a reason for the cancellation.

5. Confirm the cancellation: Once you've initiated the cancellation, you should receive a confirmation message. If you don't, double-check your order status to make sure the cancellation was successful.

If you can't find a "Cancel Order" option, it's possible that the order is no longer eligible for 

In [6]:
# ===== STEP 3.1: Load the exam + prepare the system prompt =====
import os, json
from google.colab import drive
from datasets import load_dataset

drive.mount("/content/drive")
PROJECT  = "/content/drive/MyDrive/mistral-finetune"
DATA_DIR = f"{PROJECT}/data"
RES_DIR  = f"{PROJECT}/results"
os.makedirs(RES_DIR, exist_ok=True)

# 1) Load the exam (never used in training)
test_ds = load_dataset("json", data_files=f"{DATA_DIR}/test.jsonl", split="train")
print("Exam questions loaded:", len(test_ds))
print("Intents covered      :", len(set(test_ds["intent"])))

# 2) The instruction (system prompt) for version B
SYSTEM_PROMPT = (
    "You are a friendly and professional customer support agent for an online store. "
    "Answer the customer's question politely, clearly, and with step-by-step guidance where helpful."
)

# 3) Check how Mistral receives the question WITH and WITHOUT the system prompt
sample_q = test_ds[0]["messages"][0]["content"]
msgs = [{"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": sample_q}]
print("\n----- Version A (plain) as Mistral sees it -----")
print(tokenizer.apply_chat_template(msgs[1:], tokenize=False, add_generation_prompt=True))
print("\n----- Version B (with system prompt) as Mistral sees it -----")
print(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

Exam questions loaded: 200
Intents covered      : 27

----- Version A (plain) as Mistral sees it -----
<s>[INST] help me sign up to the newsletter[/INST]

----- Version B (with system prompt) as Mistral sees it -----
<s>[INST] You are a friendly and professional customer support agent for an online store. Answer the customer's question politely, clearly, and with step-by-step guidance where helpful.

help me sign up to the newsletter[/INST]


In [7]:
# STEP 3.2: Run the exam on plain Mistral (version A)
import json, time, os, torch
from tqdm.auto import tqdm

# For batching: pad shorter questions on the LEFT so all answers start at the same point
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(question, system_prompt=None):
    msgs = [{"role": "user", "content": question}]
    if system_prompt:
        msgs = [{"role": "system", "content": system_prompt}] + msgs
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def run_exam(out_name, system_prompt=None, batch_size=8, max_new_tokens=400):

    """Answer all exam questions, saving after every batch. Resumes if interrupted."""
    out_path = f"{RES_DIR}/{out_name}.jsonl"

    # Resume: skip questions already answered in a previous run
    done = set()
    if os.path.exists(out_path):
        with open(out_path) as f:
            done = {json.loads(line)["id"] for line in f}
    todo = [i for i in range(len(test_ds)) if i not in done]
    print(f"{out_name}: {len(done)} already done, {len(todo)} to go")

    start = time.time()
    for b in tqdm(range(0, len(todo), batch_size), desc=out_name):
        ids  = todo[b:b + batch_size]
        rows = [test_ds[i] for i in ids]
        qs   = [r["messages"][0]["content"] for r in rows]

        inputs = tokenizer([build_prompt(q, system_prompt) for q in qs],
                           return_tensors="pt", padding=True,
                           add_special_tokens=False).to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id)
        new_tokens = out[:, inputs["input_ids"].shape[1]:]

        # Save this batch to Drive right away
        with open(out_path, "a") as f:
            for i, r, q, toks in zip(ids, rows, qs, new_tokens):
                f.write(json.dumps({
                    "id": i,
                    "intent": r["intent"],
                    "category": r["category"],
                    "question": q,
                    "reference": r["messages"][1]["content"],  # the ideal Bitext answer
                    "answer": tokenizer.decode(toks, skip_special_tokens=True).strip(),
                    "n_tokens": int((toks != tokenizer.pad_token_id).sum()),
                }) + "\n")

    print(f"Finished in {(time.time() - start) / 60:.1f} minutes -> {out_path}")
    return out_path

# --- Run version A: plain Mistral, no instruction ---
path_A = run_exam("baseline_A_plain", system_prompt=None)

baseline_A_plain: 0 already done, 200 to go


baseline_A_plain:   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Finished in 15.9 minutes -> /content/drive/MyDrive/mistral-finetune/results/baseline_A_plain.jsonl


In [8]:
# STEP 3.3: Run the exam on Mistral + system prompt (version B) =====
path_B = run_exam("baseline_B_system", system_prompt=SYSTEM_PROMPT)

baseline_B_system: 0 already done, 200 to go


baseline_B_system:   0%|          | 0/25 [00:00<?, ?it/s]

Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Finished in 15.8 minutes -> /content/drive/MyDrive/mistral-finetune/results/baseline_B_system.jsonl


In [9]:
# ===== STEP 3.3 VALIDATION =====
import pandas as pd
A = pd.read_json(f"{RES_DIR}/baseline_A_plain.jsonl", lines=True).sort_values("id")
B = pd.read_json(f"{RES_DIR}/baseline_B_system.jsonl", lines=True).sort_values("id")

print("Version B answers saved :", len(B))
print("Unique questions        :", B["id"].nunique())
print("Empty answers           :", (B["answer"].str.len() == 0).sum())
print("Hit 400-token limit     :", (B["n_tokens"] >= 400).sum())
print("\nAverage answer length (tokens):")
print("  A (plain)       :", int(A["n_tokens"].mean()))
print("  B (with prompt) :", int(B["n_tokens"].mean()))

# Same question, side by side
i = 5
print("\n==========", A.iloc[i]["intent"], "==========")
print("CUSTOMER   :", A.iloc[i]["question"])
print("\nA (plain)  :", A.iloc[i]["answer"][:400], "...")
print("\nB (prompt) :", B.iloc[i]["answer"][:400], "...")
print("\nIDEAL      :", A.iloc[i]["reference"][:400], "...")

Version B answers saved : 200
Unique questions        : 200
Empty answers           : 0
Hit 400-token limit     : 8

Average answer length (tokens):
  A (plain)       : 260
  B (with prompt) : 289

========== delete_account ==========
CUSTOMER   : I don't know how to close my {{Account Category}} account

A (plain)  : To close your account in a specific category, such as a bank account, credit card account, or any other type of account, you typically need to follow these general steps:

1. Gather all necessary documents and information: This may include account statements, identification documents, and any other relevant paperwork.

2. Contact the account provider: You can usually find their contact information ...

B (prompt) : Dear Customer,

Thank you for reaching out to us. I'm here to help you with your request to close your {{Account Category}} account. Here are the steps you can follow to complete this process:

1. Log in to your account on our website. If you have forgotten you

In [10]:
# ===== STEP 3.4: Read the "before" answers together =====
import pandas as pd
A = pd.read_json(f"{RES_DIR}/baseline_A_plain.jsonl", lines=True).sort_values("id").reset_index(drop=True)
B = pd.read_json(f"{RES_DIR}/baseline_B_system.jsonl", lines=True).sort_values("id").reset_index(drop=True)

# --- 1) Side by side: 5 questions from 5 different intents ---
picks = A.drop_duplicates("intent").sample(5, random_state=7).index
lines = ["# Baseline (before training) - sample answers\n"]
for i in picks:
    block = (
        f"\n## {A.loc[i,'intent']}\n"
        f"**CUSTOMER:** {A.loc[i,'question']}\n\n"
        f"**A - plain Mistral:**\n{A.loc[i,'answer']}\n\n"
        f"**B - Mistral + instruction:**\n{B.loc[i,'answer']}\n\n"
        f"**IDEAL (Bitext reference):**\n{A.loc[i,'reference']}\n"
    )
    lines.append(block)
    print(block[:1500], "\n" + "-" * 80)

with open(f"{RES_DIR}/baseline_samples.md", "w") as f:
    f.write("\n".join(lines))

# --- 2) Simple signals ---
def signals(df, name):
    ans = df["answer"].str.lower()
    ref_words = df["reference"].str.split().str.len().mean()
    return {
        "version": name,
        "uses {{placeholders}} %": round(df["answer"].str.contains(r"\{\{").mean() * 100, 1),
        "says 'as an AI'/'no access' %": round(ans.str.contains(
            r"as an ai|i'm an ai|i am an ai|language model|don't have access|do not have access|i cannot access"
        ).mean() * 100, 1),
        "avg words": int(df["answer"].str.split().str.len().mean()),
        "ideal avg words": int(ref_words),
    }

ideal = A.assign(answer=A["reference"])
summary = pd.DataFrame([signals(A, "A plain"), signals(B, "B + instruction"), signals(ideal, "IDEAL")])
print("\n===== SIMPLE SIGNALS (before training) =====")
print(summary.to_string(index=False))
summary.to_csv(f"{RES_DIR}/baseline_signals.csv", index=False)
print("\nSaved:", os.listdir(RES_DIR))


## get_invoice
**CUSTOMER:** get invoice #00108

**A - plain Mistral:**
It seems like you're asking for information related to invoice #00108. However, I don't have access to specific data unless it's provided. If you're looking for details about this invoice, I would suggest checking your records or the system where this invoice is stored. If you need help interpreting the data once you have it, feel free to ask!

**B - Mistral + instruction:**
Dear Customer,

Thank you for reaching out to us. I'm more than happy to help you with your inquiry regarding Invoice #00108.

To provide you with the necessary details, I've checked our records and found the following information:

Invoice #00108:
- Date: [Invoice Date]
- Total Amount: $[Total Amount]
- Items: [List of Items]
- Payment Method: [Payment Method]
- Status: [Invoice Status]

If you need any additional information or have questions about the items listed on this invoice, please let me know. I'm here to assist you.

Should you wish